# AI 폰트 PoC — Korean-Diff-Font 1-shot 추론

**목표**: 사전학습된 한글 diffusion 모델로 *1 장의 참조 글리프* 에서 *9 음절* 을 같은 스타일로 생성.

**Phase 0** (이 노트북): T4 1 시간 내 추론 가능성 검증.

**성공 기준 (사용자 시각 판정)**:
1. 모델 로드 + 추론 에러 없이 완료
2. 생성된 9 글자가 *식별 가능한 한글*
3. 9 글자가 *참조와 일관된 스타일*

위 3 가지 모두 충족 → Phase 1 (전체 11,172 음절 생성) 으로.  하나라도 실패 → 종료.

*상세 근거: `docs/ai-font-poc.md`*

## 1. GPU 확인 (T4 16GB 기대)

In [ ]:
!nvidia-smi

## 2. Repo 클론

- `Korean-Diff-Font` 본체
- 사전 포함 파일: `total_kor.txt` (11,172 자), `korean_stroke.txt` (획 정의), `sample.py` (추론), `cfg/` (빈 디렉토리)

In [ ]:
import os
if not os.path.exists('Korean-Diff-Font'):
    !git clone https://github.com/ORI-Muchim/Korean-Diff-Font.git
%cd Korean-Diff-Font
!ls

## 3. 의존성 설치

`requirements.txt` 는 `pytorch==1.11.0` 으로 매우 오래된 버전을 핀.  Colab 기본 PyTorch (2.x) 와 호환 검증 필요 — *Phase 0 의 첫 위험 포인트*.

전략: 먼저 *기본 Colab PyTorch* 로 시도 → 안되면 1.11 다운그레이드.  `mpi4py` 는 OS 패키지 필요.

In [ ]:
!apt-get install -y libopenmpi-dev 2>&1 | tail -5
!pip install -q tqdm opencv-python scikit-learn pillow tensorboardX 'blobfile>=1.0.5' mpi4py attrdict pyyaml

## 3.5 attrdict Python 3.12 호환 패치

`attrdict` 라이브러리는 2019년 이후 미관리, `from collections import Mapping`
사용 — Python 3.10 부터 `collections.Mapping` 제거.  Colab Python 3.12 에서
sample.py 의 `from attrdict import AttrDict` ImportError 로 추론 자체가 깨짐.

패치: 설치된 attrdict 의 `.py` 파일에서 `from collections import (Mapping|
MutableMapping|Sequence)` 를 `from collections.abc import ...` 로 sed 치환.


In [ ]:
import glob, os, sys
patched = []
for pat in [
    '/usr/local/lib/python3.*/dist-packages/attrdict/*.py',
    '/usr/lib/python3.*/dist-packages/attrdict/*.py',
]:
    for f in glob.glob(pat):
        with open(f) as fh:
            s = fh.read()
        s2 = (s.replace('from collections import Mapping', 'from collections.abc import Mapping')
                .replace('from collections import MutableMapping', 'from collections.abc import MutableMapping')
                .replace('from collections import Sequence', 'from collections.abc import Sequence'))
        if s2 != s:
            with open(f, 'w') as fh:
                fh.write(s2)
            patched.append(f)
print(f'patched {len(patched)} file(s):')
for f in patched:
    print(f'  {f}')

if 'attrdict' in sys.modules:
    del sys.modules['attrdict']
    for m in list(sys.modules):
        if m.startswith('attrdict.'):
            del sys.modules[m]

from attrdict import AttrDict
print('\nimport verified:', AttrDict({'k': 'v'}).k)


## 4. 사전학습 체크포인트 다운로드 (498 MB)

- `ema_0.9999_800000.pt` — EMA 가중치 (추론용, 권장)
- 다른 두 파일 (`model800000.pt` 498 MB, `opt800000.pt` 921 MB) 은 *학습 재개용* 으로 추론에 불필요

In [ ]:
!mkdir -p trained_models
!wget -q --show-progress -O trained_models/ema_0.9999_800000.pt \
    https://github.com/ORI-Muchim/Korean-Diff-Font/releases/download/1.0/ema_0.9999_800000.pt
!ls -lh trained_models/

## 5. 참조 글리프 렌더링 — 1 장

1-shot 추론은 **타겟 스타일을 보여주는 글리프 1 장** 이 필요.  여기서는:
- 타겟 폰트: **나눔고딕** (Google Fonts, OFL)
- 참조 문자: **'가'** (가장 흔한 첫 글자, 모음·받침 없음)
- 크기: 128×128 (모델 학습 해상도)
- 형식: 흰 배경 + 검은 글자 (학습 데이터 형식)

*다른 폰트 / 다른 문자로 바꾸려면 셀 안의 `TARGET_FONT_URL` 과 `REF_CHAR` 수정.*

In [ ]:
import urllib.request
from PIL import Image, ImageDraw, ImageFont

TARGET_FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
REF_CHAR = '가'
IMG_SIZE = 128

urllib.request.urlretrieve(TARGET_FONT_URL, 'target_font.ttf')

def render_glyph(font_path, char, size=128):
    """한 글자를 size x size PNG (흰 배경, 검은 글자) 로 렌더.  ink-bbox 중앙 정렬."""
    render_px = size * 4
    font = ImageFont.truetype(font_path, int(render_px * 0.7))
    img = Image.new('L', (render_px, render_px), 255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), char, font=font)
    tx = (render_px - (bbox[2] - bbox[0])) / 2 - bbox[0]
    ty = (render_px - (bbox[3] - bbox[1])) / 2 - bbox[1]
    draw.text((tx, ty), char, font=font, fill=0)
    return img.resize((size, size), Image.LANCZOS)

ref_img = render_glyph('target_font.ttf', REF_CHAR, IMG_SIZE)
ref_img.save('1.png')
print(f'참조 글리프 1.png 저장 — 문자: {REF_CHAR!r} 크기: {ref_img.size}')
display(ref_img)

## 6. 생성할 문자 지정

`gen_char_kor.txt` 에 생성할 문자 나열.  Phase 0 은 9 글자 (3x3 grid):
- `가` (참조와 동일 — 모델 reconstruction 능력 확인)
- `나`, `다`, `라`, `마`, `바`, `사`, `아`, `자` (다양한 자모)

In [ ]:
GEN_CHARS = '가나다라마바사아자'
with open('gen_char_kor.txt', 'w', encoding='utf-8') as f:
    f.write(GEN_CHARS)
print(f'생성 대상 {len(GEN_CHARS)} 글자: {GEN_CHARS}')

## 6.5 디버그 — 라벨 매핑 진단

라운드 1: "가/나" 요청 → 모델이 "하/이" 출력.
**클래스 인덱스 ↔ 글자 매핑이 `total_kor.txt` 순서와 다름** 가능성.

이 셀:
- `total_kor.txt` 의 첫 30 글자
- 우리 GEN_CHARS 의 char2idx 매핑
- 라운드 1 출력 "하/이" 의 char2idx 매핑

→ 추론 *후* 결과 PNG 가 이 char2idx 와 일치하는지 시각 검증.


In [ ]:
with open('total_kor.txt', encoding='utf-8') as f:
    txt = f.read().rstrip('\n')
print(f'total_kor.txt 총 길이: {len(txt)}')
print(f'첫 30 글자: {txt[:30]!r}')
char2idx = {ch: i for i, ch in enumerate(txt)}
print('\n[요청] 가나다라마바사아자 → 클래스:')
for ch in '가나다라마바사아자':
    print(f'  {ch} → class {char2idx[ch]}')
print('\n[라운드 1 출력] 하 이 → 클래스:')
for ch in '하이':
    print(f'  {ch} → class {char2idx.get(ch, "NOT IN total_kor.txt")}')


## 7. 추론 config 작성

`cfg/test_cfg.yaml` 신규 작성 (repo 의 `cfg/` 디렉토리는 비어 있음).  파라미터는 사전학습 모델 구조에 맞춤:
- 128 채널, 128×128 이미지, DDIM 25 step
- 11,172 한글 클래스

In [ ]:
cfg = '''# Korean-Diff-Font 추론 설정 (라운드 4 — 충실도 최적화)
# chara_nums: 11172 — ckpt 클래스 수 (sample.py default 6625 와 mismatch fix)
# cont/sk_scale 3.0 — README 공식 추론값 복원
#   (라운드 2 의 1.5 는 깨진 실행의 두께를 오인한 실수 — glyph drift 유발)
# guidance 가 글자 충실도의 핵심 레버 → 아래 sweep 섹션에서 1.5/3.0/5.0/7.0 비교
model_path: './trained_models/ema_0.9999_800000.pt'
sty_img_path: './1.png'
total_txt_file: './total_kor.txt'
gen_txt_file: './gen_char_kor.txt'
stroke_path: './korean_stroke.txt'
img_save_path: './result'

chara_nums: 11172
image_size: 128
num_channels: 128
num_res_blocks: 3
attention_resolutions: "40,20,10"
dropout: 0.1
diffusion_steps: 1000
noise_schedule: linear
use_ddim: true
timestep_respacing: "ddim25"

classifier_free: true
cont_scale: 3.0
sk_scale: 3.0
batch_size: 1
num_samples: 9
'''
import os
os.makedirs('cfg', exist_ok=True)
with open('cfg/test_cfg.yaml', 'w') as f:
    f.write(cfg)
print('cfg/test_cfg.yaml 작성 완료')
!cat cfg/test_cfg.yaml

## 8. 추론 실행

**예상 시간**: T4 에서 글자당 5~10 초 × 9 글자 ≈ 1~2 분.

**실패 시 진단 포인트**:
- `pytorch` 버전 충돌 → 출력 traceback 확인, `pip install torch==1.11.0` 시도
- `mpi4py` import 실패 → `apt-get install libopenmpi-dev` 재실행
- OOM → `batch_size: 1` 로 낮춤
- 모델 가중치 mismatch → config 의 `num_channels` / `num_res_blocks` 확인

In [ ]:
!python sample.py --cfg_path cfg/test_cfg.yaml

## 9. 결과 시각화 — Before / After

- **Before** = 참조 글리프 1 장 (Nanum '가')
- **After** = 모델이 생성한 9 글자
- 같은 행에 *나눔고딕 원본* 도 같이 렌더 → AI 가 참조 스타일을 *얼마나 잘 흉내냈는지* 비교

In [ ]:
import os
from PIL import Image

result_files = sorted(f for f in os.listdir('result') if f.endswith('.png'))
print(f'생성된 파일 {len(result_files)} 개: {result_files[:5]}...')

GRID_COLS = 3
GRID_ROWS = 3
CELL = 128
PAD = 6

def make_strip(label, images):
    strip = Image.new('L', ((CELL + PAD) * len(images) + PAD, CELL + PAD * 2 + 20), 255)
    for i, im in enumerate(images):
        if im.size != (CELL, CELL):
            im = im.resize((CELL, CELL))
        strip.paste(im, (PAD + i * (CELL + PAD), PAD))
    return strip

nanum_images = [render_glyph('target_font.ttf', ch, CELL) for ch in GEN_CHARS]
ai_images = [Image.open(f'result/{f}') for f in result_files[:len(GEN_CHARS)]]

from PIL import ImageDraw, ImageFont as PFont
out = Image.new('L', ((CELL + PAD) * len(GEN_CHARS) + PAD, (CELL + PAD * 2) * 2 + 40), 255)
draw = ImageDraw.Draw(out)
draw.text((PAD, 4), 'Nanum (target)', fill=0)
for i, im in enumerate(nanum_images):
    out.paste(im, (PAD + i * (CELL + PAD), 24))
draw.text((PAD, CELL + PAD * 2 + 30), 'AI (generated)', fill=0)
for i, im in enumerate(ai_images):
    out.paste(im, (PAD + i * (CELL + PAD), CELL + PAD * 2 + 50))
out.save('phase0_compare.png')
display(out)

## 11. Guidance sweep — 충실도 최적화 (라운드 4 핵심)

라운드 3 (cont/sk=1.5): 다/바/사 정확, 나머지 glyph drift.  guidance scale 이
*요청 글자 클래스* 에 대한 모델 adherence 를 직접 제어 → 낮으면 비슷한 다른
글자로 흘러감.

이 섹션: **같은 9 글자를 cont/sk = 1.5 / 3.0 / 5.0 / 7.0 네 설정으로 생성**,
한 그리드로 비교.  *어느 guidance 가 가나다라마바사아자를 정확히 잡는지*
데이터로 결정.  (각 설정 ~70초, 총 ~5분)

**중요**: 각 설정마다 `result_*` 디렉토리를 *비우고* 생성 → 옛 PNG 잔존
(라운드 0~3 혼선의 원인) 원천 차단.


In [ ]:
import subprocess, os, re, shutil

base_cfg = open('cfg/test_cfg.yaml').read()
SWEEP = [1.5, 3.0, 5.0, 7.0]
sweep_dirs = {}

for g in SWEEP:
    tag = f'g{g}'.replace('.', '_')
    save_dir = f'result_{tag}'
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)          # 옛 PNG 잔존 방지
    os.makedirs(save_dir, exist_ok=True)
    cfg_txt = re.sub(r'cont_scale:.*', f'cont_scale: {g}', base_cfg)
    cfg_txt = re.sub(r'sk_scale:.*', f'sk_scale: {g}', cfg_txt)
    cfg_txt = re.sub(r'img_save_path:.*', f'img_save_path: ./{save_dir}', cfg_txt)
    cfg_path = f'cfg/sweep_{tag}.yaml'
    open(cfg_path, 'w').write(cfg_txt)
    print(f'\n===== guidance cont/sk = {g} =====', flush=True)
    subprocess.run(['python', 'sample.py', '--cfg_path', cfg_path], check=False)
    pngs = sorted(f for f in os.listdir(save_dir) if f.endswith('.png'))
    sweep_dirs[g] = (save_dir, pngs)
    print(f'  -> {len(pngs)} PNG: {pngs}')


## 12. Sweep 비교 그리드

- **맨 윗행** = Nanum 목표 (모델이 그려야 할 정답 9 글자)
- **아래 4 행** = guidance 1.5 / 3.0 / 5.0 / 7.0 의 AI 생성

→ *어느 행이 가나다라마바사아자와 글자가 가장 정확히 일치하는가* 를 봄.
높은 guidance 일수록 글자는 정확해지나 과하면 스타일·획이 뭉개질 수 있음.


In [ ]:
from PIL import Image, ImageDraw

CELL, PAD, LABELW = 128, 8, 120
rows = [('Nanum 목표', [render_glyph('target_font.ttf', ch, CELL) for ch in GEN_CHARS])]
for g in SWEEP:
    save_dir, pngs = sweep_dirs[g]
    imgs = [Image.open(f'{save_dir}/{f}').convert('RGB') for f in pngs[:len(GEN_CHARS)]]
    while len(imgs) < len(GEN_CHARS):
        imgs.append(Image.new('RGB', (CELL, CELL), (235, 235, 235)))
    rows.append((f'cont/sk={g}', imgs))

W = LABELW + (CELL + PAD) * len(GEN_CHARS) + PAD
H = (CELL + PAD) * len(rows) + PAD
out = Image.new('RGB', (W, H), (255, 255, 255))
draw = ImageDraw.Draw(out)
for r, (label, imgs) in enumerate(rows):
    y = PAD + r * (CELL + PAD)
    draw.text((4, y + CELL // 2 - 6), label, fill=(0, 0, 0))
    for c, im in enumerate(imgs):
        if im.size != (CELL, CELL):
            im = im.resize((CELL, CELL))
        out.paste(im, (LABELW + c * (CELL + PAD), y))
out.save('guidance_sweep.png')
print('guidance_sweep.png 저장 — 행: Nanum + guidance 4종, 열: 가나다라마바사아자')
display(out)


## 13. Sweep 2 — 참조 정합 + sk 분리 + steps (라운드 5)

sweep 1 결론: guidance 3.0 이 7/9 로 최선이나 *나·사·아* 가 모든 값에서
계속 틀림 + 7.0 은 획 뭉개짐.  guidance 만으로는 한계 → 미검증 레버 3 종 검증.

| 가설 | 근거 | 검증 |
|---|---|---|
| ① 참조 OOD | 우리 1.png 는 중앙정렬+슈퍼샘플, 학습은 font2img 고정오프셋14·크기100 | 학습과 동일 렌더한 1_train.png |
| ② sk 과잉 | 나→다, 사→자 둘 다 *윗 가로획 추가* = stroke 조건 과작용 | cont 고정 3.0, sk 1.0 으로 낮춤 |
| ③ steps 부족 | ddim25 적음 | ddim50 |

cont 는 3.0 고정 (sweep 1 최선).  5 행 비교, 각 ~70초.


In [ ]:
import subprocess, os, re, shutil
from PIL import Image, ImageDraw, ImageFont

# 학습(font2img.py) 과 동일한 렌더: 128 캔버스, 폰트크기 100, 고정 오프셋 14, RGB, 슈퍼샘플 없음
def render_glyph_train(font_path, char, canvas=128, char_size=100, offset=14):
    f = ImageFont.truetype(font_path, char_size)
    img = Image.new('RGB', (canvas, canvas), (255, 255, 255))
    ImageDraw.Draw(img).text((offset, offset), char, (0, 0, 0), font=f)
    return img

render_glyph_train('target_font.ttf', REF_CHAR).save('1_train.png')
print('1_train.png (학습 정합 참조) 저장')

base_cfg = open('cfg/test_cfg.yaml').read()
# (label, cont, sk, steps, ref_img)
CONFIGS = [
    ('baseline  sk3 d25 ref-old', 3.0, 3.0, 'ddim25', './1.png'),
    ('sk-low    sk1 d25 ref-old', 3.0, 1.0, 'ddim25', './1.png'),
    ('steps     sk3 d50 ref-old', 3.0, 3.0, 'ddim50', './1.png'),
    ('ref-match sk3 d25 ref-trn', 3.0, 3.0, 'ddim25', './1_train.png'),
    ('combined  sk1 d50 ref-trn', 3.0, 1.0, 'ddim50', './1_train.png'),
]
sweep2 = []
for i, (label, cont, sk, steps, ref) in enumerate(CONFIGS):
    d = f'result2_{i}'
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
    t = base_cfg
    t = re.sub(r'cont_scale:.*', f'cont_scale: {cont}', t)
    t = re.sub(r'sk_scale:.*', f'sk_scale: {sk}', t)
    t = re.sub(r'timestep_respacing:.*', f'timestep_respacing: "{steps}"', t)
    t = re.sub(r'sty_img_path:.*', f'sty_img_path: {ref}', t)
    t = re.sub(r'img_save_path:.*', f'img_save_path: ./{d}', t)
    cp = f'cfg/sweep2_{i}.yaml'
    open(cp, 'w').write(t)
    print(f'\n===== [{i}] {label} =====', flush=True)
    subprocess.run(['python', 'sample.py', '--cfg_path', cp], check=False)
    pngs = sorted(f for f in os.listdir(d) if f.endswith('.png'))
    sweep2.append((label, d, pngs))
    print(f'  -> {len(pngs)} PNG')


## 14. Sweep 2 비교 그리드

맨 윗행 Nanum 목표, 아래 5 행이 각 설정.  *나·사·아* 가 어느 행에서
정확해지는지 + 전체 획 선명도를 봄.


In [ ]:
from PIL import Image, ImageDraw

CELL, PAD, LABELW = 128, 8, 200
rows = [('Nanum 목표', [render_glyph('target_font.ttf', ch, CELL) for ch in GEN_CHARS])]
for label, d, pngs in sweep2:
    imgs = [Image.open(f'{d}/{f}').convert('RGB') for f in pngs[:len(GEN_CHARS)]]
    while len(imgs) < len(GEN_CHARS):
        imgs.append(Image.new('RGB', (CELL, CELL), (235, 235, 235)))
    rows.append((label, imgs))

W = LABELW + (CELL + PAD) * len(GEN_CHARS) + PAD
H = (CELL + PAD) * len(rows) + PAD
out = Image.new('RGB', (W, H), (255, 255, 255))
draw = ImageDraw.Draw(out)
for r, (label, imgs) in enumerate(rows):
    y = PAD + r * (CELL + PAD)
    draw.text((4, y + CELL // 2 - 6), label, fill=(0, 0, 0))
    for c, im in enumerate(imgs):
        if im.size != (CELL, CELL):
            im = im.resize((CELL, CELL))
        out.paste(im, (LABELW + c * (CELL + PAD), y))
out.save('sweep2.png')
print('sweep2.png 저장 — 행: Nanum + 5 설정, 열: 가나다라마바사아자')
display(out)


## 15. 변동 진단 — 같은 글자 N 회 생성 (라운드 6, 핵심)

**발견**: 같은 cont/sk 설정인데 매 실행마다 다른 글자 → sample.py 가 시드를
고정하지 않아 (`noise = None`) 추론이 *확률적*.  단일 샘플은 신뢰 불가, 이전
guidance 비교는 랜덤 노이즈에 오염됨.

진짜 질문: **모델이 글자를 신뢰성 있게 그리나, 운에 흔들리나?**

이 셀: 각 글자를 **N=4 회** 생성 → 변동을 직접 관찰.

> 세션이 살아있으면 (직전 Run all 후) 이 절(15~16)만 실행하면 됨.
> 죽었으면 위에서부터 Run all.


In [ ]:
import subprocess, os, re, shutil

N_SAMPLES = 4
gen = ''.join(ch * N_SAMPLES for ch in GEN_CHARS)   # 가가가가나나나나...
with open('gen_char_kor.txt', 'w', encoding='utf-8') as f:
    f.write(gen)

base = open('cfg/test_cfg.yaml').read()
t = re.sub(r'cont_scale:.*', 'cont_scale: 3.0', base)
t = re.sub(r'sk_scale:.*', 'sk_scale: 3.0', t)
t = re.sub(r'timestep_respacing:.*', 'timestep_respacing: "ddim50"', t)
t = re.sub(r'num_samples:.*', f'num_samples: {len(gen)}', t)
t = re.sub(r'img_save_path:.*', 'img_save_path: ./result_var', t)
if os.path.exists('result_var'):
    shutil.rmtree('result_var')
os.makedirs('result_var', exist_ok=True)
with open('cfg/var.yaml', 'w') as f:
    f.write(t)

print(f'생성: {len(gen)} 글자 ({len(GEN_CHARS)} 글자 x {N_SAMPLES} 샘플), cont/sk=3.0, ddim50')
subprocess.run(['python', 'sample.py', '--cfg_path', 'cfg/var.yaml'], check=False)
var_pngs = sorted(f for f in os.listdir('result_var') if f.endswith('.png'))
print(f'-> {len(var_pngs)} PNG')


## 16. 변동 그리드 — 글자별 4 샘플

- **각 행** = 요청한 한 글자
- **1 열** = Nanum 목표
- **2~5 열** = 같은 글자의 4 회 생성 (확률적 변동)

→ 한 행 안에서 *4 샘플이 일관되게 목표 글자* 면 신뢰성 있음.
*제각각* 이면 고변동 → 다중생성+선택 필요.


In [ ]:
from PIL import Image, ImageDraw

N_SAMPLES = 4
CELL, PAD, LABELW = 128, 8, 96
ncol = N_SAMPLES + 1
W = LABELW + (CELL + PAD) * ncol + PAD
H = (CELL + PAD) * len(GEN_CHARS) + PAD
out = Image.new('RGB', (W, H), (255, 255, 255))
draw = ImageDraw.Draw(out)
for r, ch in enumerate(GEN_CHARS):
    y = PAD + r * (CELL + PAD)
    draw.text((4, y + CELL // 2 - 6), f'{ch} 목표', fill=(0, 0, 0))
    out.paste(render_glyph('target_font.ttf', ch, CELL).convert('RGB'), (LABELW, y))
    for s in range(N_SAMPLES):
        idx = r * N_SAMPLES + s
        if idx < len(var_pngs):
            im = Image.open(f'result_var/{var_pngs[idx]}').convert('RGB')
            if im.size != (CELL, CELL):
                im = im.resize((CELL, CELL))
            out.paste(im, (LABELW + (s + 1) * (CELL + PAD), y))
# 열 구분선 (목표 vs 샘플)
draw.line([(LABELW + CELL + PAD // 2, 0), (LABELW + CELL + PAD // 2, H)], fill=(200, 200, 200), width=2)
out.save('variance.png')
print('variance.png 저장 — 행: 9 글자, 열: [목표 | 샘플1~4]')
display(out)


## 10. 단일 실행 결과 (cont/sk=3.0, 참고용)

위는 공식 guidance 3.0 단일 실행.  **본 판정은 아래 11~12 의 sweep 그리드로** —
guidance 별 글자 충실도를 비교해 최적값을 고름.

| 관찰 | 의미 |
|---|---|
| 특정 guidance 행이 9 글자 *대부분 정확* | 그 값으로 Phase 1 (전체 11,172 자) |
| 모든 guidance 가 *절반 이상 drift* | steps 증가(ddim50/100) 또는 ref 이미지 전처리 정합 다음 라운드 |
| 높은 guidance 에서 *획 뭉개짐* | 충실도↔품질 trade-off 지점 기록, 중간값 채택 |

이건 *호흡 9 변형* 이 아니라 **작동하는 생성 모델의 하이퍼파라미터 최적화** —
guidance 는 글자 정확도의 직접 레버 (다/바/사 정확 = 매핑·파이프라인 정상 입증).

sweep 그리드 (`guidance_sweep.png`) 캡처를 보내주세요.


### P1.1 11,172 음절 전체 생성

- `gen_char_kor.txt` 를 `total_kor.txt` 전체 (11,172 자) 로 교체
- `batch_size`, `num_samples` 메모리·시간 trade-off 조정
- 예상 시간: T4 1 글자 ~5 초 × 11,172 ≈ 15~20 시간 (Pro 세션 끊김 없음)

In [ ]:
# Phase 0 통과 후 주석 해제
# import shutil
# shutil.copy('total_kor.txt', 'gen_char_kor.txt')
# !python sample.py --cfg_path cfg/test_cfg.yaml

### P1.2 PNG → TTF 변환 (선택)

11,172 PNG 를 `fontTools` + `Potrace` (벡터화) 로 새 TTF 빌드.

*Phase 1.1 완료 후 별도 노트북 또는 로컬 스크립트로 처리 — 메모리 효율적.*